In [ ]:
import os
import numpy as np

def load_and_stack_csv_pair(mech_csv_path, bio_csv_path):
    array_mech = np.loadtxt(mech_csv_path, delimiter=',')
    array_bio = np.loadtxt(bio_csv_path, delimiter=',')
    # Expand dimensions to prepare for channel stacking
    array_mech = np.expand_dims(array_mech, axis=-1)
    array_bio = np.expand_dims(array_bio, axis=-1)
    # Stack along the channel axis (Depth = 2)
    return np.concatenate((array_mech, array_bio), axis=-1)

dataset_dir = "/content/drive/MyDrive/Skin_Cancer_Dataset"
categories = {"Normal": 0, "Scar": 1, "Inflamed": 2, "Tumor": 3}

X_data, Y_labels = [], []
print("Starting data loading process...")

for category_name, label_value in categories.items():
    folder_path = os.path.join(dataset_dir, category_name)
    if not os.path.exists(folder_path): continue

    all_files = os.listdir(folder_path)
    mech_files = [f for f in all_files if "_mech_values_values.csv" in f]

    loaded_count = 0
    for mech_filename in mech_files:
        bio_filename = mech_filename.replace("_mech_values_values.csv", "_bio_values_values.csv")
        mech_path = os.path.join(folder_path, mech_filename)
        bio_path = os.path.join(folder_path, bio_filename)

        if os.path.exists(bio_path):
            stacked_tensor = load_and_stack_csv_pair(mech_path, bio_path)
            X_data.append(stacked_tensor)
            Y_labels.append(label_value)
            loaded_count += 1

    print(f"Loaded {loaded_count} samples from {category_name}.")

X_data = np.array(X_data)
Y_labels = np.array(Y_labels)
print(f"Master Dataset Shape: {X_data.shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import numpy as np

def attention_block(inputs):
    channels = inputs.shape[-1]
    x = layers.GlobalAveragePooling2D()(inputs)
    x = layers.Dense(channels // 4, activation='relu')(x)
    x = layers.Dense(channels, activation='sigmoid')(x)
    x = layers.Reshape((1, 1, channels))(x)
    channel_refined = layers.Multiply()([inputs, x])

    y = layers.Conv2D(1, (3, 3), padding='same', activation='sigmoid')(channel_refined)
    spatial_refined = layers.Multiply()([channel_refined, y])
    return spatial_refined

def build_attention_cnn(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Orthogonal spatial augmentation strictly preserves 3x3 topological limits
    x = layers.RandomFlip("horizontal_and_vertical")(inputs)

    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = attention_block(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = attention_block(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Flatten strictly preserves topological placement of sensors
    x = layers.Flatten()(x)
    x = layers.Dense(64, use_bias=False)(x)
    x = layers.BatchNormalization(name='deep_features')(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(4, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)

    # 5% Label smoothing prevents overconfidence on boundary interpolation noise
    loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    return model

def create_standard_dataset(X, Y, batch_size=16):
    Y_one_hot = tf.one_hot(Y, depth=4)
    ds = tf.data.Dataset.from_tensor_slices((X, Y_one_hot))
    ds = ds.shuffle(1024).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def predict_with_tta(model, x_test):
    predictions = [
        model.predict(x_test, verbose=0),
        model.predict(tf.image.flip_left_right(x_test), verbose=0),
        model.predict(tf.image.flip_up_down(x_test), verbose=0),
        model.predict(tf.image.rot90(x_test, k=1), verbose=0),
        model.predict(tf.image.rot90(x_test, k=3), verbose=0)
    ]
    return np.mean(predictions, axis=0)

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from tensorflow.keras import callbacks

output_dir = "/content/drive/MyDrive/Skin_Cancer_Dataset/Publication_Data"
os.makedirs(output_dir, exist_ok=True)
print(f"Publication data will be saved to: {output_dir}")

mech_oof_probs, bio_oof_probs, fused_oof_probs = [], [], []
oof_true_labels = []
acc_mech, acc_bio, acc_fused = [], [], []

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# THE OPTIMIZATION: Smooth Cosine Annealing
def cosine_schedule(epoch, lr):
    epochs = 80
    lr_init = 0.001
    lr_min = 0.00001
    return lr_min + 0.5 * (lr_init - lr_min) * (1 + np.cos(np.pi * epoch / epochs))

lr_decay_callback = callbacks.LearningRateScheduler(cosine_schedule)

print("\nExecuting Cosine-Optimized Ablation 5-Fold Cross-Validation...")
fold_no = 1
for train_index, test_index in kfold.split(X_data, Y_labels):
    print(f"\n--- Processing Fold {fold_no} ---")

    X_train_fold, X_test_fold = X_data[train_index], X_data[test_index]
    Y_train_fold, Y_test_fold = Y_labels[train_index], Y_labels[test_index]

    X_train_mech, X_test_mech = X_train_fold[:, :, :, 0:1], X_test_fold[:, :, :, 0:1]
    X_train_bio, X_test_bio = X_train_fold[:, :, :, 1:2], X_test_fold[:, :, :, 1:2]

    # --- MODEL 1: BIOMECHANICAL ONLY ---
    print("Training Biomechanical-Only Baseline...")
    model_mech = build_attention_cnn(input_shape=(96, 96, 1))
    model_mech.fit(create_standard_dataset(X_train_mech, Y_train_fold), epochs=80, callbacks=[lr_decay_callback], verbose=0)
    probs_m = predict_with_tta(model_mech, X_test_mech)
    mech_oof_probs.extend(probs_m)
    acc_mech.append(np.mean(np.argmax(probs_m, axis=1) == Y_test_fold) * 100)
    tf.keras.backend.clear_session()

    # --- MODEL 2: BIOCHEMICAL ONLY ---
    print("Training Biochemical-Only Baseline...")
    model_bio = build_attention_cnn(input_shape=(96, 96, 1))
    model_bio.fit(create_standard_dataset(X_train_bio, Y_train_fold), epochs=80, callbacks=[lr_decay_callback], verbose=0)
    probs_b = predict_with_tta(model_bio, X_test_bio)
    bio_oof_probs.extend(probs_b)
    acc_bio.append(np.mean(np.argmax(probs_b, axis=1) == Y_test_fold) * 100)
    tf.keras.backend.clear_session()

    # --- MODEL 3: MULTIMODAL FUSION ---
    print("Training Multimodal Fusion Platform...")
    model_fused = build_attention_cnn(input_shape=(96, 96, 2))
    model_fused.fit(create_standard_dataset(X_train_fold, Y_train_fold), epochs=80, callbacks=[lr_decay_callback], verbose=0)
    probs_f = predict_with_tta(model_fused, X_test_fold)
    fused_oof_probs.extend(probs_f)
    acc_fused.append(np.mean(np.argmax(probs_f, axis=1) == Y_test_fold) * 100)
    tf.keras.backend.clear_session()

    oof_true_labels.extend(Y_test_fold)
    fold_no += 1

print(f"\n=======================================================")
print(f"ABLATION RESULTS (5-Fold CV Accuracy):")
print(f"Biomechanical-Only Baseline: {np.mean(acc_mech):.2f}%")
print(f"Biochemical-Only Baseline:   {np.mean(acc_bio):.2f}%")
print(f"Multimodal Fusion Platform:  {np.mean(acc_fused):.2f}%")
print(f"=======================================================")

print("\nExporting Raw Publication Data CSVs to Google Drive...")
class_names = ['Normal', 'Scar', 'Inflamed', 'Tumor']

df_roc = pd.DataFrame({'True_Label': oof_true_labels})
for i, c in enumerate(class_names):
    df_roc[f'Mech_Prob_{c}'] = np.array(mech_oof_probs)[:, i]
    df_roc[f'Bio_Prob_{c}'] = np.array(bio_oof_probs)[:, i]
    df_roc[f'Fused_Prob_{c}'] = np.array(fused_oof_probs)[:, i]

df_roc['Fused_Predicted_Label'] = np.argmax(np.array(fused_oof_probs), axis=1)
df_roc.to_csv(os.path.join(output_dir, "Ablation_ROC_Probabilities.csv"), index=False)

print("Pipeline complete. CV probabilities secured for figure generation.")